# Wind Farm Life Cycle Assessment

**Run the cells in order:**
1. Install & import
2. Load database
3. Configure (widgets)
4. Run simulation
5. Result display functions
6. *(result cells — run each individually after step 5)*

## 1 · Install & import

Installs required packages and loads all input files, configuration, and engine functions into memory.

In [ ]:
import subprocess, sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install",
     "pandas", "openpyxl", "matplotlib", "pyyaml", "ipywidgets",
     "--quiet"],
    stdout=subprocess.DEVNULL
)

import yaml, os
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

# ── Load configuration files ───────────────────────────────────────────────
_here = os.getcwd()

with open(os.path.join(_here, "inputs", "inventory_codes.yaml"), encoding="utf-8") as _f:
    INVENTORY_CODES = yaml.safe_load(_f)

with open(os.path.join(_here, "inputs", "config.yaml"), encoding="utf-8") as _f:
    IMPACT_COLUMNS = yaml.safe_load(_f)["IMPACT_COLUMNS"]

from inventory_masses import INVENTORY_MASSES
from lca_engine import (
    load_ecoinvent_database,
    resolve_emission_factors,
    merge_inventories,
    calculate_impacts,
    aggregate_by_stage,
    validate_inventory_coverage,
)

print("Setup complete.")

## 2 · Load database

Reads the ecoinvent Excel file into memory and verifies that every UID in the inventory is present in the database.

In [ ]:
print("Loading ecoinvent database...")
ecoinvent_db, n_processes = load_ecoinvent_database()
print(f"Loaded {n_processes} processes.")

missing = validate_inventory_coverage(INVENTORY_CODES, ecoinvent_db)
if missing:
    print(f"\n[NOTE] {len(missing)} UID(s) not found in the database:")
    for comp_name, life_stage, code in missing:
        print(f"  - {comp_name} ({life_stage}): {code}")
else:
    print("All UIDs found in ecoinvent database.")

## 3 · Configure

Tick the life cycle stages and scope(s) to include. All stages are selected by default.

In [ ]:
_all_stages = list(INVENTORY_CODES.keys())

stage_checkboxes = {
    stage: widgets.Checkbox(value=True, description=stage,
                            layout=widgets.Layout(width="340px"))
    for stage in _all_stages
}

scope_checkboxes = {
    "per_turbine": widgets.Checkbox(value=True,  description="Per Turbine"),
    "full_farm":   widgets.Checkbox(value=True,  description="Full Farm"),
    "per_FU":      widgets.Checkbox(value=False, description="Per Functional Unit (kWh)"),
}

display(widgets.VBox([
    widgets.HTML("<b>Life cycle stages:</b>"),
    *stage_checkboxes.values(),
    widgets.HTML("<br><b>Scope(s):</b>"),
    *scope_checkboxes.values(),
]))

## 4 · Run simulation

Merges inventory masses with ecoinvent emission factors and calculates GWP100 for every component. Results are stored silently — use the cells below to display them.

In [ ]:
selected_stages = [stage for stage, cb in stage_checkboxes.items() if cb.value]
selected_scopes = [s for s, cb in scope_checkboxes.items() if cb.value]

if not selected_stages:
    raise ValueError("No life cycle stages selected — tick at least one above and re-run.")
if not selected_scopes:
    raise ValueError("No scope selected — tick at least one above and re-run.")

print(f"Stages : {', '.join(selected_stages)}")
print(f"Scopes : {', '.join(selected_scopes)}")
print("\nResolving emission factors...")
resolved_efs = resolve_emission_factors(INVENTORY_CODES, selected_stages, ecoinvent_db)
print(f"Resolved {len(resolved_efs)} unique UIDs.")

results = {}
for scope in selected_scopes:
    print(f"\nCalculating: {scope.replace('_', ' ').title()}...")
    merged = merge_inventories(INVENTORY_CODES, INVENTORY_MASSES, selected_stages, scope)
    if not merged:
        print(f"  [WARNING] No components matched for {scope} — skipping.")
        continue
    impact_results = calculate_impacts(merged, resolved_efs)
    if not impact_results:
        print(f"  [WARNING] No impacts calculated for {scope} — skipping.")
        continue
    results[scope] = aggregate_by_stage(impact_results)
    print(f"  Done — {len(impact_results)} components with valid impacts.")

print("\nSimulation complete. Run the cells below to view results.")

## 5 · Result display functions

Defines all table and chart functions used by the result cells below. Must be run once before any result cell.

In [ ]:
def _gwp_label():
    return list(IMPACT_COLUMNS.keys())[0]


def print_summary_table(aggregated_results, scope):
    by_stage    = aggregated_results["by_stage"]
    grand_total = aggregated_results["grand_total"]
    gwp         = _gwp_label()
    scope_label = scope.replace("_", " ").title()

    print("\n" + "=" * 56)
    print(f"  GWP100 RESULTS — {scope_label}")
    print("=" * 56)
    print(f"  {'Life Stage':<28} {'kg CO2-Eq':>22}")
    print("  " + "-" * 52)
    for stage, impacts in by_stage.items():
        print(f"  {stage:<28} {impacts.get(gwp, 0.0):>22.4e}")
    print("  " + "-" * 52)
    print(f"  {'TOTAL':<28} {grand_total.get(gwp, 0.0):>22.4e}")
    print("=" * 56 + "\n")


def print_full_emissions_table(aggregated_results, scope):
    gwp         = _gwp_label()
    components  = aggregated_results["by_component"]
    by_stage    = aggregated_results["by_stage"]
    grand_total = aggregated_results["grand_total"]
    scope_label = scope.replace("_", " ").title()

    stages_map = {}
    for comp in components:
        stage = comp["life_stage"]
        stages_map.setdefault(stage, []).append(comp)

    W_STAGE = 18
    W_COMP  = 38
    W_QTY   = 14
    W_UNIT  =  6
    W_EF    = 18
    W_TOT   = 18
    SEP     = "  " + "-" * (W_STAGE + W_COMP + W_QTY + W_UNIT + W_EF + W_TOT + 11)
    WIDTH   = W_STAGE + W_COMP + W_QTY + W_UNIT + W_EF + W_TOT + 13

    print("\n" + "=" * WIDTH)
    print(f"  FULL EMISSIONS BREAKDOWN — {scope_label}")
    print("=" * WIDTH)
    print(
        f"  {'Life Stage':<{W_STAGE}} "
        f"{'Component':<{W_COMP}} "
        f"{'Quantity':>{W_QTY}} "
        f"{'Unit':<{W_UNIT}} "
        f"{'EF (kg CO2/unit)':>{W_EF}} "
        f"{'kg CO2-Eq':>{W_TOT}}"
    )
    print(SEP)

    for stage, comps in stages_map.items():
        stage_label    = stage
        stage_subtotal = by_stage.get(stage, {}).get(gwp, 0.0)
        for comp in comps:
            quantity = comp.get("quantity", 0.0) or 0.0
            unit     = comp.get("unit", "")
            gwp_val  = comp.get(gwp, 0.0)
            ef       = (gwp_val / quantity) if quantity else 0.0
            name     = comp["component_name"][:W_COMP]
            print(
                f"  {stage_label:<{W_STAGE}} "
                f"{name:<{W_COMP}} "
                f"{quantity:>{W_QTY}.4e} "
                f"{unit:<{W_UNIT}} "
                f"{ef:>{W_EF}.4e} "
                f"{gwp_val:>{W_TOT}.4e}"
            )
            stage_label = ""
        subtotal_label = f"  Subtotal — {stage}"
        print(f"  {'':>{W_STAGE}} {subtotal_label:<{W_COMP + W_QTY + W_UNIT + W_EF + 3}} {stage_subtotal:>{W_TOT}.4e}")
        print(SEP)

    print(
        f"  {'GRAND TOTAL':<{W_STAGE}} "
        f"{'':>{W_COMP + W_QTY + W_UNIT + W_EF + 3}} "
        f"{grand_total.get(gwp, 0.0):>{W_TOT}.4e}"
    )
    print("=" * WIDTH + "\n")


def plot_materials_gwp(aggregated_results, scope):
    gwp         = _gwp_label()
    components  = aggregated_results["by_component"]
    scope_label = scope.replace("_", " ").title()

    group_totals = {}
    for comp in components:
        if comp["life_stage"] != "Materials":
            continue
        top_group = comp["path"].split(" > ")[0]
        group_totals[top_group] = group_totals.get(top_group, 0.0) + comp.get(gwp, 0.0)

    if not group_totals:
        print("  [INFO] No Materials data available for chart — skipping.")
        return

    groups = list(group_totals.keys())
    values = list(group_totals.values())

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(groups, values, color=plt.cm.Set2.colors[:len(groups)],
                  edgecolor="white", linewidth=0.8)
    ax.set_xlabel("Component Group", fontsize=12)
    ax.set_ylabel("GWP100 (kg CO2-Eq)", fontsize=12)
    ax.set_title(f"Materials Stage — GWP100 by Component Group\n({scope_label})",
                 fontsize=13, fontweight="bold")
    ax.tick_params(axis="x", rotation=20)
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                    f"{val:.2e}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def plot_stage_gwp_pie(aggregated_results, scope):
    gwp         = _gwp_label()
    by_stage    = aggregated_results["by_stage"]
    scope_label = scope.replace("_", " ").title()

    stage_vals = {stage: impacts.get(gwp, 0.0) for stage, impacts in by_stage.items()}
    positive   = {k: v for k, v in stage_vals.items() if v > 0}
    negative   = {k: v for k, v in stage_vals.items() if v < 0}

    if not positive:
        print("  [INFO] No positive GWP stage data — skipping pie chart.")
        return

    labels  = list(positive.keys())
    values  = list(positive.values())
    total   = sum(values)
    palette = ["#4C72B0", "#DD8452", "#55A868", "#C44E52",
               "#8172B3", "#937860", "#DA8BC3", "#8C8C8C",
               "#CCB974", "#64B5CD"]

    fig, ax = plt.subplots(figsize=(9, 7))
    wedges, _, autotexts = ax.pie(
        values, labels=None,
        autopct=lambda pct: f"{pct:.1f}%" if pct >= 2 else "",
        startangle=90, colors=palette[:len(labels)],
        wedgeprops=dict(width=0.55, edgecolor="white", linewidth=1.5),
        pctdistance=0.75,
    )
    for at in autotexts:
        at.set_fontsize(9)
        at.set_fontweight("bold")
        at.set_color("white")
    ax.text(0, 0, f"Total\n{total:.2e}\nkg CO2-Eq",
            ha="center", va="center", fontsize=10, fontweight="bold", color="#333333")
    ax.legend(
        wedges, [f"{lbl}  ({v:.2e} kg CO2-Eq)" for lbl, v in zip(labels, values)],
        loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=9, frameon=False
    )
    ax.set_title(f"GWP100 Contribution by Life Stage\n({scope_label})",
                 fontsize=13, fontweight="bold", pad=20)
    if negative:
        fig.text(0.5, 0.01,
                 "Net credits (not in chart): " +
                 "  |  ".join(f"{s}: {v:.2e}" for s, v in negative.items()) +
                 " kg CO2-Eq",
                 ha="center", fontsize=8, color="#666666")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


print("Result functions loaded.")

---
## Results

Run each cell below individually to display its output.
> **Note:** cells 4 and 5 (simulation + result functions) must be run first.

### 6 · GWP100 summary table

Total GWP100 (kg CO₂-eq) broken down by life stage, with a grand total, for each selected scope.

In [ ]:
try:
    results
except NameError:
    print("[INFO] Run the simulation cell first (cell 4).")
else:
    for scope, aggregated in results.items():
        print_summary_table(aggregated, scope)

### 7 · Full emissions breakdown

Component-level detail showing quantity, unit, emission factor, and kg CO₂-eq for every item, grouped by life stage with subtotals.

In [ ]:
try:
    results
except NameError:
    print("[INFO] Run the simulation cell first (cell 4).")
else:
    for scope, aggregated in results.items():
        print_full_emissions_table(aggregated, scope)

### 8 · Materials stage — GWP100 by component group (bar chart)

Bar chart comparing the GWP100 contribution of each top-level component group within the Materials life stage.

In [ ]:
try:
    results
except NameError:
    print("[INFO] Run the simulation cell first (cell 4).")
else:
    for scope, aggregated in results.items():
        plot_materials_gwp(aggregated, scope)

### 9 · GWP100 contribution by life stage (donut chart)

Donut chart showing each life stage's percentage share of the total GWP100. Net credits (negative values) are listed below the chart.

In [ ]:
try:
    results
except NameError:
    print("[INFO] Run the simulation cell first (cell 4).")
else:
    for scope, aggregated in results.items():
        plot_stage_gwp_pie(aggregated, scope)